In [2]:
import nltk
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [3]:
nltk.download('punkt')

# Load pre-trained language model
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Makai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\Makai\AppData\Roaming\Python\Python312\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
# Function to measure coherence
def get_coherence(text, window_size=4096):
    # Tokenize text
    tokens = tokenizer("\n" + text + "\n", return_tensors='pt', truncation=True, max_length=window_size)
    input_ids = tokens.input_ids
    
    # Split the text into smaller contexts (short and long contexts)
    short_context = input_ids[:, :window_size // 2]
    # long_context = input_ids
    
    # Predict on short and long contexts
    with torch.no_grad():
        outputs_short = model(short_context, labels=short_context)
        # outputs_long = model(long_context, labels=long_context)
    
    # Calculate accuracy and log-likelihood difference
    coherence_acc_short = (outputs_short.logits.argmax(-1) == short_context).float().mean().item()
    # coherence_acc_long = (outputs_long.logits.argmax(-1) == long_context).float().mean().item()
    
    #coherence_diff = (coherence_acc_long - coherence_acc_short) / coherence_acc_long
    return coherence_acc_short


In [26]:
from rdflib import Graph, Namespace, URIRef, Literal


In [22]:
# Assuming get_coherence function is provided
def get_coherence(text: str) -> float:
    # This is a placeholder for the actual coherence calculation
    return len(text) / 10.0  # Simplified for demonstration

In [40]:
def process_ttl_file(input_file, output_file):
    # Create an RDF graph
    graph = Graph()

    # Parse the input TTL file
    graph.parse(input_file, format='ttl')

    # Define the namespace
    EX = Namespace("http://example.org/")

    # Iterate through all triples to find ex:Token subjects linked to other ex:Tokens
    tokens = set()
    for s, p, o in graph:
        if isinstance(s, URIRef) and isinstance(o, URIRef) and s.startswith(EX) and o.startswith(EX):
            if s != o:  # Ensure we're not linking a token to itself
                tokens.add((s, o))

    # Add new links with the label as the coherence difference between nodes
    for s, o in tokens:
        coherence_s = get_coherence(str(s))
        coherence_o = get_coherence(str(o))
        coherence_difference = abs(coherence_s - coherence_o)
        graph.add((s, EX.coherence_diff, Literal(coherence_difference)))

    # Serialize the graph to a new TTL file
    graph.serialize(destination=output_file, format='ttl')
    print(f"Graph successfully written to {output_file}")

In [41]:
# Define the input and output file names
input_ttl_file = 'test9.ttl'  # Replace with your actual input file name
output_ttl_file = 'coherence_graph.ttl'

# Process the TTL file
process_ttl_file(input_ttl_file, output_ttl_file)

Graph successfully written to coherence_graph.ttl
